# Lab 4: Graph features in models, link prediction, embeddings and a look at GNNs

**You will:**
1. recommend new payment relationships the way social networks recommend friends
2. add graph features to an ordinary churn model and measure the lift (the workhorse of graph ML)
3. build node embeddings with FastRP in a few lines, and learn what they do and don't capture
4. see that a graph neural network layer is "average your neighbours, then multiply by a matrix", and where that idea pays off

In [ ]:
# Setup
import numpy as np, pandas as pd, networkx as nx, scipy.sparse as sp
import plotly.graph_objects as go
from IPython.display import display
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import labkit as lk

nodes, edges, truth = lk.load(truth=True)
name, sector, region = (dict(zip(nodes.node_id, nodes[c])) for c in ("name", "sector", "region"))
HUBS = lk.find_hubs(edges)
businesses = lambda g: g.subgraph([n for n in g if n.startswith("B")]).copy()
display(f"{len(HUBS)} hubs removed in every graph below")

## 1. Link prediction: "People You May Know" for payments

Social networks recommend friends with **triadic closure**: if A knows B and B knows C, A and C are likely to meet. The score is just a count of shared neighbours, and Adamic-Adar gives more credit to shared neighbours who are themselves selective.

Here: learn on the first half-year, then check which **new** relationships actually appeared in the second half.

In [ ]:
# Train on H1, test on relationships that are new in H2
g1 = businesses(lk.payment_graph(edges, lk.MONTHS[:6], directed=False, drop=HUBS))
g2 = lk.payment_graph(edges, lk.MONTHS[6:], directed=False, drop=HUBS)
new = {frozenset((u, v)) for u, v in g2.edges() if u in g1 and v in g1 and not g1.has_edge(u, v)}

cand = set()                                    # every pair two hops apart and not yet linked
for x in g1:
    nb = set(g1[x])
    for y in nb:
        cand.update((x, z) for z in g1[y] if z != x and z not in nb and x < z)
cand = list(cand)
y = np.array([frozenset(c) in new for c in cand])

scores = {
    "common neighbours": [len(set(g1[a]) & set(g1[b])) for a, b in cand],
    "Adamic-Adar": [s for _, _, s in nx.adamic_adar_index(g1, cand)],
    "Jaccard": [s for _, _, s in nx.jaccard_coefficient(g1, cand)],
    "preferential attachment": [s for _, _, s in nx.preferential_attachment(g1, cand)],
    "random": np.random.default_rng(0).random(len(cand)),
}
prec = {k: y[np.argsort(-np.asarray(v))[:200]].mean() for k, v in scores.items()}
fig = lk.bar(list(prec), [round(v, 3) for v in prec.values()],
             f"Precision of the top 200 suggestions (random pair: {y.mean():.2%})",
             ylabel="share that really appeared in H2", text=[f"{v:.1%}" for v in prec.values()], height=400)
fig.update_yaxes(tickformat=".0%")
display(f"{len(new):,} new relationships in H2; {y.sum():,} of them were two hops apart in H1 "
        f"({y.sum() / len(new):.0%}). Candidates scored: {len(cand):,}.", fig)

Shared neighbours beat random by more than an order of magnitude, and most suggestions are still wrong: new relationships are hard to predict. In practice this becomes a ranked "businesses like you also pay" list for a relationship manager, not an automatic decision.

## 2. Graph features in an ordinary model: the workhorse

Which businesses will stop transacting in the next three months? Same model (gradient boosting), three feature sets:

1. **account only**: own amounts and payment counts, and their trend
2. **+ one hop**: number of distinct payers and payees, and their trend, top-counterparty share
3. **+ network**: what is happening to the **counterparties** (are they gone, are they cutting their own partners?), PageRank, clustering, the community's trend

Cutoff: features use data up to 2026-05; the label is "went silent in 2026-06 to 2026-08".

In [ ]:
# Build the three feature sets
CUT = "2026-05"
base, recent = lk.MONTHS[:6], lk.MONTHS[6:9]
e = edges[(edges.month <= CUT) & ~edges.src.isin(HUBS) & ~edges.dst.isin(HUBS)]
biz = sorted(set(nodes.node_id[nodes.party_type == "O"]) - set(HUBS))
active_at_cut = set(e[e.month == CUT].src) | set(e[e.month == CUT].dst)
pop = [b for b in biz if b in active_at_cut]
out = e.rename(columns={"src": "node", "dst": "cp"})
inn = e.rename(columns={"dst": "node", "src": "cp"})

def per_month(df, months, ids):
    d = df[df.month.isin(months)]
    g = d.groupby(["node", "month"]).agg(amt=("amount", "sum"), ntx=("n_txn", "sum"), ncp=("cp", "nunique")).reset_index()
    return (g.groupby("node")[["amt", "ntx", "ncp"]].sum() / len(months)).reindex(ids).fillna(0)

A_ = pd.DataFrame(index=pop)          # account only
H_ = pd.DataFrame(index=pop)          # + one hop
for side, df in (("out", out), ("in", inn)):
    b, r = per_month(df, base, pop), per_month(df, recent, pop)
    A_[f"{side}_amount_trend"] = np.log((r.amt + 100) / (b.amt + 100))      # log ratio: 0 = flat
    A_[f"{side}_payments_trend"] = np.log((r.ntx + .5) / (b.ntx + .5))
    A_[f"{side}_amount_recent"] = np.log1p(r.amt)
    H_[f"{side}_partners_trend"] = np.log((r.ncp + .25) / (b.ncp + .25))
    H_[f"{side}_partners_recent"] = r.ncp
    last = df[df.month == CUT].groupby(["node", "cp"]).amount.sum()
    H_[f"top_{side}_share"] = (last.groupby("node").max() / last.groupby("node").sum()).reindex(pop).fillna(0)
A_["average_payment"] = np.log1p(per_month(out, recent, pop).amt / per_month(out, recent, pop).ntx.replace(0, np.nan)).fillna(0)

# network: what is happening to my counterparties?
silent = set(biz) - active_at_cut
ob, orr = per_month(out, base, biz), per_month(out, recent, biz)
cp_trend = (orr.ncp + .25) / (ob.ncp + .25)
w = pd.concat([out, inn])[lambda d: d.month.isin(base)].groupby(["node", "cp"]).amount.sum().reset_index()
w["gone"] = w.cp.isin(silent)
w["cutting"] = w.cp.map(cp_trend).fillna(1) < 0.75
tot = w.groupby("node").amount.sum()
N_ = pd.DataFrame(index=pop)
N_["partners_gone_share"] = ((w.amount * w.gone).groupby(w.node).sum() / tot).reindex(pop).fillna(0)
N_["partners_cutting_share"] = ((w.amount * w.cutting).groupby(w.node).sum() / tot).reindex(pop).fillna(0)
gb = businesses(lk.payment_graph(e, base, directed=False))
pr, cl = nx.pagerank(gb, weight="weight"), nx.clustering(gb)
N_["pagerank"] = [pr.get(x, 0) for x in pop]
N_["clustering"] = [cl.get(x, 0) for x in pop]
comm = {v: i for i, c in enumerate(nx.community.louvain_communities(gb, weight="weight", resolution=2.0, seed=1)) for v in c}
cdf = pd.DataFrame({"c": [comm.get(x, -1) for x in biz], "b": ob.amt.values, "r": orr.amt.values,
                    "gone": [x in silent for x in biz]}, index=biz)
cg = cdf.groupby("c").agg(b=("b", "sum"), r=("r", "sum"), gone=("gone", "mean"))
cg["trend"] = np.log((cg.r + 100) / (cg.b + 100))
N_["community_trend"] = [cg.trend[comm.get(x, -1)] for x in pop]
N_["community_gone_share"] = [cg.gone[comm.get(x, -1)] for x in pop]

label = truth.set_index("node_id").churn_label.reindex(pop).values
display(f"{len(pop):,} businesses active at {CUT}; {label.sum()} ({label.mean():.1%}) go silent in the next three months",
        pd.concat([A_, H_, N_], axis=1).describe().T[["mean", "50%", "max"]].round(3))

In [ ]:
# Train and compare (5-fold cross-validation)
try:
    import lightgbm as lgb
    make = lambda: lgb.LGBMClassifier(n_estimators=250, learning_rate=0.03, num_leaves=7, min_child_samples=30,
                                      subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1, random_state=0)
    engine = "LightGBM"
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier
    make = lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=250, random_state=0)
    engine = "sklearn HistGradientBoosting"

sets = {"account only": A_, "+ one hop": A_.join(H_), "+ network": A_.join(H_).join(N_)}
cv = StratifiedKFold(5, shuffle=True, random_state=0)
res, preds = [], {}
for k, X in sets.items():
    p = cross_val_predict(make(), X, label, cv=cv, method="predict_proba")[:, 1]
    preds[k] = p
    top = np.argsort(-p)[:len(p) // 10]
    res.append({"features": k, "n features": X.shape[1], "AUC": roc_auc_score(label, p),
                "churners caught in the top 10%": label[top].sum() / label.sum()})
res = pd.DataFrame(res)
fig = go.Figure([go.Bar(x=res.features, y=res.AUC, marker=dict(color=lk.PALETTE[0]), text=res.AUC.round(3),
                        textposition="outside")])
fig.update_yaxes(range=[0.5, 1], title="AUC (cross-validated)")
display(res.round(3), lk.style(fig, f"Same model ({engine}), more graph: the AUC ladder", height=400))

In [ ]:
# Which features does the full model lean on?
m = make().fit(sets["+ network"], label)
imp = pd.Series(getattr(m, "feature_importances_", np.zeros(sets["+ network"].shape[1])),
                index=sets["+ network"].columns).sort_values()
family = ["account" if c in A_ else "one hop" if c in H_ else "network" for c in imp.index]
colour = {"account": lk.OTHER, "one hop": lk.PALETTE[0], "network": lk.PALETTE[1]}
fig = go.Figure()
for f in ("account", "one hop", "network"):
    m_ = [i for i, x in enumerate(family) if x == f]
    fig.add_trace(go.Bar(y=imp.index[m_], x=imp.values[m_], orientation="h", name=f, marker=dict(color=colour[f])))
fig.update_layout(barmode="stack", yaxis=dict(categoryorder="array", categoryarray=list(imp.index)))
lk.style(fig, "Feature importance: partner trends lead; partners' health (orange) adds what the account cannot see", height=560)

**The lesson.** The biggest lift comes from cheap one-hop features: how many partners a customer has, and whether that number is falling. Network features add more on top: a customer whose partners are leaving or cutting back is at risk *before* its own numbers move. That last part is invisible to any model that only sees the customer.

## 3. Embeddings: FastRP in a few lines

An **embedding** gives every node a vector so that similar nodes get similar vectors. **FastRP** (fast random projection, 2019) needs no training:

1. give every node a random sparse vector
2. replace each vector by the average of its neighbours' vectors, a few times
3. add up the steps

Every step is a sparse matrix multiply, the same operation as a join plus a group-by, so it scales to Spark.

In [ ]:
# FastRP
G = businesses(lk.payment_graph(edges, directed=False, drop=HUBS))
order = list(G)
A = nx.to_scipy_sparse_array(G, nodelist=order, weight="weight").tocsr()

def fastrp(A, dim=64, weights=(0.0, 1.0, 1.0, 1.0), seed=0):
    n = A.shape[0]
    S = sp.diags(1 / np.maximum(np.asarray(A.sum(1)).ravel(), 1e-12)) @ A       # average over neighbours
    R = np.random.default_rng(seed).choice([np.sqrt(3), 0, -np.sqrt(3)], size=(n, dim), p=[1 / 6, 2 / 3, 1 / 6])
    X, emb = R, weights[0] * R
    for wgt in weights[1:]:
        X = S @ X
        X = X / np.linalg.norm(X, axis=1, keepdims=True).clip(1e-12)
        emb = emb + wgt * X
    return emb

Z = fastrp(A)
xy = TSNE(n_components=2, random_state=0, init="pca", perplexity=40).fit_transform(Z)
comm = {v: i for i, c in enumerate(sorted(nx.community.louvain_communities(G, weight="weight", resolution=2.0, seed=1),
                                          key=len, reverse=True)) for v in c}
fig = go.Figure(go.Scatter(x=xy[:, 0], y=xy[:, 1], mode="markers",
                           marker=dict(size=5, color=[lk.PALETTE[comm[v]] if comm[v] < 8 else lk.OTHER for v in order]),
                           hovertext=[f"{name[v]}<br>{region[v]}, {sector[v]}" for v in order], hoverinfo="text"))
lk._layout(fig, f"FastRP embeddings ({Z.shape[1]} numbers per business), squeezed to 2-D; colour = community", 600, legend=False)

### What embeddings capture, and what they don't

Take a pass-through account from Lab 2 and ask for its five nearest neighbours: first in FastRP space, then using the one-hop **role** measures from Lab 2 (degrees, throughput, overlap, individual payer share).

In [ ]:
# Nearest neighbours: proximity (FastRP) vs role (local measures)
arch = truth.set_index("node_id").archetype
Zn = Z / np.linalg.norm(Z, axis=1, keepdims=True)
ix = {v: i for i, v in enumerate(order)}

win = edges[edges.month.isin(lk.MONTHS[-3:]) & ~edges.src.isin(HUBS) & ~edges.dst.isin(HUBS)]
pairs = win.groupby(["src", "dst"], as_index=False).amount.sum()
o_ = pairs.groupby("src").agg(out_deg=("dst", "nunique"), out_amt=("amount", "sum"), top=("amount", "max"))
i_ = pairs.groupby("dst").agg(in_deg=("src", "nunique"), in_amt=("amount", "sum"),
                              ind=("src", lambda s: s.str.startswith("P").sum()))
back = pairs.merge(pairs, left_on=["src", "dst"], right_on=["dst", "src"], suffixes=("", "_b"))
L = i_.join(o_, how="outer").fillna(0)
L = L[L.index.str.startswith("B")]
L["rec"] = back.groupby("src").dst.nunique().reindex(L.index).fillna(0)
role = pd.DataFrame({
    "log_in": np.log1p(L.in_deg), "log_out": np.log1p(L.out_deg),
    "throughput": (np.minimum(L.in_amt, L.out_amt) / np.maximum(L.in_amt, L.out_amt).replace(0, np.nan)).fillna(0),
    "overlap": (L.rec / L[["in_deg", "out_deg"]].min(1).replace(0, np.nan)).fillna(0),
    "individual_payers": (L.ind / L.in_deg.replace(0, np.nan)).fillna(0),
    "log_volume": np.log1p(L.in_amt + L.out_amt), "top_payee": (L.top / L.out_amt.replace(0, np.nan)).fillna(0)})
Rn = StandardScaler().fit_transform(role)
Rn = Rn / np.linalg.norm(Rn, axis=1, keepdims=True)
rix = {v: i for i, v in enumerate(role.index)}

probe = next(v for v in order if arch[v] == "pass_through")
nn_fast = [order[i] for i in np.argsort(-(Zn @ Zn[ix[probe]]))[1:6]]
nn_role = [role.index[i] for i in np.argsort(-(Rn @ Rn[rix[probe]]))[1:6]]
display(pd.DataFrame({
    "FastRP neighbour": [f"{v} {name[v]}" for v in nn_fast], "its planted type": [arch[v] for v in nn_fast],
    "role neighbour": [f"{v} {name[v]}" for v in nn_role], "its planted type ": [arch[v] for v in nn_role]}))

hit = {}
for a in ("pass_through", "funnel", "round_trip"):
    members = [v for v in role.index if arch[v] == a]
    hit[a] = np.mean([np.mean([arch[role.index[i]] == a for i in np.argsort(-(Rn @ Rn[rix[v]]))[1:6]]) for v in members])
display(pd.Series(hit, name="share of 5 role neighbours with the same planted type").round(2).to_frame())

**FastRP answers "who is near you"** (same ecosystem, same partners). **Role features answer "who behaves like you"**: the nearest neighbours of a pass-through by role are the other pass-throughs. A look-alike search (for example "find customers who behave like our registered payment processors") needs role features, or embeddings trained on them, not proximity alone.

## 4. A glimpse of graph neural networks

One GNN layer (GraphSAGE style) does two things for every node:

1. **aggregate**: average the neighbours' vectors
2. **update**: concatenate with the node's own vector, multiply by a learned weight matrix, apply a non-linearity

Two layers mean each node sees two hops. Training learns the weight matrices from labels.

In [ ]:
# A two-layer GraphSAGE forward pass in numpy (random weights: this is the shape of the computation)
S = sp.diags(1 / np.maximum(np.asarray(A.sum(1)).ravel(), 1e-12)) @ A      # mean over neighbours
X0 = StandardScaler().fit_transform(role.reindex(order).fillna(0))          # 7 features per business
rng = np.random.default_rng(0)
W1 = rng.normal(0, 0.3, (2 * X0.shape[1], 16))
W2 = rng.normal(0, 0.3, (2 * 16, 8))
relu = lambda x: np.maximum(x, 0)
H1 = relu(np.hstack([X0, S @ X0]) @ W1)       # layer 1: me + mean of my neighbours -> 16 numbers
H2 = relu(np.hstack([H1, S @ H1]) @ W2)       # layer 2: now each node has seen two hops -> 8 numbers
pd.DataFrame({"step": ["input features", "after layer 1", "after layer 2"],
              "shape": [X0.shape, H1.shape, H2.shape],
              "what it is": ["own measures", "own + neighbours' average, mixed by W1", "own + two-hop neighbourhood, mixed by W2"]})

**Where the idea pays off: filling in missing attributes.** Hide the sector of 30% of businesses and predict it. Labels come free (every business with a known sector is a training example), which is exactly when graph learning earns its keep. Simplified graph convolution (SGC, 2019) is a GNN without the non-linearity, so plain logistic regression can train it.

In [ ]:
# Predict a hidden sector: own features vs neighbours' sectors vs both (SGC)
secs = pd.Series({v: sector[v] for v in order})
top = secs.value_counts().index[:10]
yv = secs.where(secs.isin(top), "Other sectors")
hide = np.random.default_rng(1).random(len(order)) < 0.3
Gd = lk.payment_graph(edges, directed=True)
own = np.column_stack([[float(Gd.has_edge(v, h)) for v in order] for h in HUBS] +
                      [[float(Gd.has_edge(h, v)) for v in order] for h in HUBS] +
                      [np.log1p([G.degree(v) for v in order])])         # which hubs it pays or is paid by
Y = pd.get_dummies(yv).astype(float).values
Yk = Y * (~hide)[:, None]                                               # only the visible sectors

def accuracy(F):
    m = LogisticRegression(max_iter=3000).fit(F[~hide], yv[~hide])
    return (m.predict(F[hide]) == yv[hide]).mean()

vote = np.array(pd.get_dummies(yv).columns)[(S @ Yk).argmax(1)]
acc = pd.Series({
    "guess the most common sector": (yv[hide] == yv[~hide].mode()[0]).mean(),
    "own features only": accuracy(own),
    "neighbours' sectors (vote)": (vote[hide] == yv[hide].values).mean(),
    "own + neighbours (SGC)": accuracy(np.hstack([own, S @ own, S @ Yk, S @ (S @ Yk)])),
})
fig = lk.bar(list(acc.index), acc.values.round(3), "Accuracy on the hidden 30%: combining a node with its neighbourhood wins",
             ylabel="accuracy", text=[f"{v:.0%}" for v in acc.values], height=420)
fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig

With PyTorch Geometric installed, the same model is a few lines (`SAGEConv`, `GCNConv`) and trains on mini-batches of sampled neighbourhoods. The decision of **whether** to use one matters more than the library:

* worth it: attribute imputation with free labels (industry, location, entity type), next-month link prediction
* rarely worth it: decision models with scarce or rule-derived labels, mostly degree-1 nodes, monthly snapshots, and a model-risk reviewer who asks "why was this customer flagged?"
* the cheap test: add the embeddings to the existing model and see whether the AUC moves

## Takeaways

* Triadic closure (shared neighbours) is a strong, simple baseline for recommending new relationships.
* Graph features in ordinary models are the workhorse; one-hop features carry most of the lift, partners' health adds what the customer's own data cannot show.
* FastRP embeddings capture proximity; role features capture behaviour. Choose by the question.
* A GNN layer is neighbour averaging plus a learned matrix: most useful where labels come free.